# Research, Demo Checks, and Analytics

This notebook walks through sample-data research, demo-account connectivity, and real-data perp analytics.

In [1]:
import os

os.chdir('..')
os.listdir()

['.DS_Store',
 'pyproject.toml',
 'tests',
 'README.md',
 'configs',
 'data',
 'notebooks',
 'src']

In [5]:
from src.bybit_trader.exchange import BybitClient

BYBIT_DEMO_API_KEY = "GG9MlzsDKVfAg0rQcW"
BYBIT_DEMO_API_SECRET = "SqUMpULjz4OroxRSgZnxdnkLo5gotoNy31d9"

demo_client = BybitClient(api_key=BYBIT_DEMO_API_KEY, api_secret=BYBIT_DEMO_API_SECRET, testnet=True)
# demo_client.get_wallet_balance()

In [6]:
from src.bybit_trader.session import NotebookSession

session = NotebookSession(bybit_client=demo_client).with_sample_data()
session.list_strategies()

['carry_basket',
 'momentum_rotation',
 'option_iv_hv_long_gamma',
 'option_premium_fade',
 'perp_mean_reversion',
 'perp_trend',
 'protective_option_overlay',
 'volatility_breakout']

In [7]:
sample_research = session.research(['BTCUSDT', 'ETHUSDT', 'SOLUSDT', 'XRPUSDT'])
sample_backtest = session.backtest('perp_trend')
(sample_research, sample_backtest.summary())

({'BTCUSDT': {'last_price': 81608.12954697378,
   'period_return': 0.052173679912468796,
   'avg_volume': 1601.4830321023007,
   'avg_funding': -3.1298831970132656e-05},
  'ETHUSDT': {'last_price': 4202.739384534373,
   'period_return': 0.018793504178279186,
   'avg_volume': 1707.615607368323,
   'avg_funding': -2.8182460506966892e-05},
  'SOLUSDT': {'last_price': 206.22495453357757,
   'period_return': -0.17637862247672076,
   'avg_volume': 1840.3979547949411,
   'avg_funding': -1.259015720987018e-05},
  'XRPUSDT': {'last_price': 0.770887981743331,
   'period_return': 0.038465190442629194,
   'avg_volume': 1948.3297610247437,
   'avg_funding': -3.0457098719020673e-05}},
 {'ending_equity': 91042.32,
  'total_return': -0.089577,
  'max_drawdown': 0.161173,
  'trade_count': 745.0,
  'avg_trade_pnl': -17.072914})

## Demo Account Connectivity

This uses the local demo credential source automatically, so the notebook stays free of inline secrets.

In [8]:
demo_session = NotebookSession.from_config('./configs/default.toml').attach_default_demo_client()
{'demo': demo_session.bybit_client.demo, 'testnet': demo_session.bybit_client.testnet}

RuntimeError: Demo credentials are not configured. Add the local credential source or set BYBIT_DEMO_API_KEY and BYBIT_DEMO_API_SECRET.

In [10]:
from src.bybit_trader.verification import run_verification_suite

[entry.as_dict() for entry in run_verification_suite('private-demo-topics')]

[{'name': 'verify-private-demo-topics',
  'status': 'failed',
  'message': 'Missing demo credentials. Add the local demo credential source or set BYBIT_DEMO_API_KEY and BYBIT_DEMO_API_SECRET.',
  'details': {},
  'duration_seconds': 0.001}]

## Real Bybit Data and Backtest Analytics

Fetch 180 days of 1h perp data for BTC, ETH, SOL, and one dynamically discovered liquid alt, then run the default perp strategy set.

In [12]:
real_session = NotebookSession.from_config('./configs/default.toml')
import_summary = real_session.fetch_real_market_data(lookback_days=180, interval_minutes=60)
import_summary

{'symbols': ['BTCUSDT', 'ETHUSDT', 'SOLUSDT', 'DOGEUSDT'],
 'bar_counts': {'BTCUSDT': 4320,
  'ETHUSDT': 4320,
  'SOLUSDT': 4320,
  'DOGEUSDT': 4320},
 'funding_counts': {'BTCUSDT': 540,
  'ETHUSDT': 540,
  'SOLUSDT': 540,
  'DOGEUSDT': 540}}

In [13]:
analytics_rows = real_session.run_default_perp_analytics(symbols=import_summary['symbols'])
analytics_rows

[{'strategy_name': 'perp_trend',
  'universe': ['BTCUSDT', 'ETHUSDT', 'SOLUSDT', 'DOGEUSDT'],
  'ending_equity': 94444.13,
  'total_return': -0.055559,
  'max_drawdown': 0.347061,
  'trade_count': 4625,
  'average_trade_pnl': -0.503263,
  'win_rate': 0.301405,
  'gross_turnover': 15604632.98,
  'turnover_ratio': 156.04633,
  'average_gross_exposure': 78110.84,
  'funding_contribution': 60.356542,
  'best_symbol': 'ETHUSDT',
  'best_symbol_pnl': 554.98968,
  'worst_symbol': 'SOLUSDT',
  'worst_symbol_pnl': -3953.252937,
  'symbol_contributions': {'BTCUSDT': -2496.163127,
   'DOGEUSDT': 338.557187,
   'ETHUSDT': 554.98968,
   'SOLUSDT': -3953.252937},
  'notes': ''},
 {'strategy_name': 'perp_mean_reversion',
  'universe': ['BTCUSDT', 'ETHUSDT', 'SOLUSDT', 'DOGEUSDT'],
  'ending_equity': 100000.0,
  'total_return': 0.0,
  'max_drawdown': 0.0,
  'trade_count': 0,
  'average_trade_pnl': 0.0,
  'win_rate': 0.0,
  'gross_turnover': 0,
  'turnover_ratio': 0.0,
  'average_gross_exposure': 0,
  

In [ ]:
option_research = real_session.option_research_summary()
option_research